# QLoRA fine-tune -- Qwen2.5-3B

**Run this in Colab on a T4 (free) or an A100.** Nothing here works on a laptop: it needs a CUDA GPU
for 4-bit quantisation.

The plan: load a 3B base model in 4-bit nf4, attach LoRA adapters, and train on the tasting-note
prompts so the model completes `Price is $` with a number. Only the adapters train, which is what
makes this fit in 16GB.

Sweep variant: attention projections only, per-device batch 8 with accumulation
2 -- the same effective batch as the canonical run. The difference measured here is what the FFN
adapters contribute.

Add two Colab secrets first (key icon, left sidebar): `HF_TOKEN` from
[huggingface.co](https://huggingface.co/settings/tokens) and `WANDB_API_KEY` from
[wandb.ai](https://wandb.ai/authorize). The run streams to Weights & Biases, which is how you watch a
multi-hour fine-tune without leaving the tab open.

In [ ]:
!pip install -q -U \
    "transformers>=4.56.2" \
    "peft>=0.17" \
    "trl>=1.0" \
    "bitsandbytes>=0.44" \
    "torchao>=0.16.0" \
    "datasets>=3.0" \
    "accelerate>=1.0" \
    "wandb>=0.18"
!git clone -q https://github.com/borjahernandez/wine-pricer.git
%cd wine-pricer

In [ ]:
import math
import os
from datetime import datetime

import torch
import torchao
import wandb
from datasets import load_dataset
from google.colab import userdata
from huggingface_hub import login
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

from pricer.prompts import as_completion

login(userdata.get("HF_TOKEN"))

BASE_MODEL = "Qwen/Qwen2.5-3B"
DATASET = "borjahernandez/wine-pricer"

PROJECT_NAME = "wine-pricer"
HF_USER = "borjahernandez"
RUN = "wine-pricer-qwen3b"
RUN_NAME = f"{RUN}-{datetime.now():%Y%m%d-%H%M}"  # one W&B run per attempt, so sweeps stay legible
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Tracking
VAL_SIZE = 500
LOG_STEPS = 5
SAVE_STEPS = 100

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"  # adapters go to the Hub; W&B only needs the curves
os.environ["WANDB_WATCH"] = "false"  # gradient histograms cost throughput and rarely answer anything
wandb.login()

capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8
DTYPE = torch.bfloat16 if use_bf16 else torch.float16

In [ ]:
# The Hub dataset carries every curated field; the fine-tune reads one column pair. Splitting
# `prompt` at `Price is $` leaves the question as `prompt` and the bare price as `completion`, which is
# what `completion_only_loss` masks against -- the model is scored on the number, never on the prose.
# VAL_SIZE validation rows are scored every SAVE_STEPS, so overfitting shows up mid-run.
data = load_dataset(DATASET)
train = data["train"].map(as_completion, input_columns="prompt", remove_columns=data["train"].column_names)
val = (
    data["validation"]
    .map(as_completion, input_columns="prompt", remove_columns=data["validation"].column_names)
    .select(range(VAL_SIZE))
)

print(train)
print(train[0])

### Hyperparameters

| knob | value | why |
| --- | --- | --- |
| `r` | 32 | adapter rank. 8 underfits here, 64 costs memory for little gain |
| `alpha` | 64 | conventionally 2r |
| target modules | attention only | half the capacity -- measures what the FFN adapters buy |
| `lr` | 1e-4 | LoRA tolerates rates ~10x a full fine-tune |
| effective batch | 8 x 2 = 16 | held constant across the sweep |
| gradient checkpointing | off | trades recompute for memory |
| epochs | 1 | 80k examples is plenty; a second epoch mostly memorises |
| 4-bit nf4, double quant | on | the whole reason this fits on a T4 |

In [ ]:
LORA = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

QUANT = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=DTYPE,
)

STEPS = math.ceil(len(train) / 16)  # 79,359 / 16 = 4,960 optimizer steps

CONFIG = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=1,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=int(0.03 * STEPS),
    optim="adamw_torch_fused",
    weight_decay=0.001,
    max_grad_norm=0.3,
    max_length=256,
    completion_only_loss=True,
    gradient_checkpointing=False,
    fp16=not use_bf16,
    bf16=use_bf16,
    report_to="wandb",
    run_name=RUN_NAME,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    push_to_hub=True,
    hub_strategy="every_save",
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=QUANT,
    device_map="auto",
    dtype=DTYPE,
)
model.generation_config.pad_token_id = tokenizer.pad_token_id

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
model.config.use_cache = False

print(f"Memory footprint: {model.get_memory_footprint() / 1e6:.1f} MB")

trainer = SFTTrainer(
    model=model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=LORA,
    args=CONFIG,
)

# PEFT creates the adapters in bf16 (it follows the model config, which says bfloat16 regardless
# of the load dtype). The fp16 GradScaler cannot unscale bf16 gradients, so cast to fp32 --
# which is what QLoRA does anyway.
for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

In [ ]:
# Labels are built at map time now, and a prompt longer than `max_length` is dropped rather than
# truncated -- silently, since there is no exception and no loss spike to notice. Long notes are
# written about expensive bottles, so any loss lands in the thin top bins the balancing protects.
dropped = len(train) - len(trainer.train_dataset)
assert not dropped, f"{dropped} rows exceeded max_length={CONFIG.max_length} and were dropped"

trainer.train()
trainer.push_to_hub(f"Fine-tuned on {DATASET}")
wandb.finish()  # without this the run stays live and the summary metrics never settle

### Score it on the full test split

The pushed adapter is loaded back onto the quantized base and evaluated on all 2,000 test wines --
the same set, the same `evaluate`, the same `results.json` as every other model. Two details keep
the row comparable with the ModernBERT encoder's:

- **Predictions are clamped to [0, $1,000]**, the same bound the encoder applies in log space.
  Unbounded, one hallucinated "4500" would dominate the error and the comparison stops being
  about pricing.
- **Parse failures are counted.** A generative model can emit anything; the encoder structurally
  cannot fail here, so the rate is a real cost of the decoder approach.

In [ ]:
import re

from peft import PeftModel

from pricer.evaluator import evaluate, leaderboard
from pricer.items import Wine

ADAPTER = "borjahernandez/wine-pricer-wine-pricer-qwen3b-20260901-1636"  # the adapter repo this run pushed to the Hub

# Must match the ModernBERT evaluation exactly, or the two leaderboard rows are not comparable.
MAX_LENGTH = 256
MAX_PRICE = 1000.0

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=QUANT,
    device_map="auto",
    dtype=DTYPE,
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

model = PeftModel.from_pretrained(base_model, ADAPTER)
model.eval()

_, _, test = Wine.from_hub(DATASET)

failures = 0


def parse_price(text: str) -> float:
    match = re.search(r"[-+]?\d[\d,]*\.?\d*", text.replace("$", ""))
    if not match:
        return 0.0
    return min(max(float(match.group().replace(",", "")), 0.0), MAX_PRICE)


def specialist(wine: Wine) -> float:
    global failures
    inputs = tokenizer(
        wine.test_prompt(), return_tensors="pt", truncation=True, max_length=MAX_LENGTH
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=6, do_sample=False)
    completion = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:])
    price = parse_price(completion)
    if price == 0.0:
        # A generative model can emit anything. ModernBERT structurally cannot fail here,
        # so this rate is a real cost of the decoder approach and belongs in the write-up.
        failures += 1
    return price


specialist.__name__ = "Qwen2.5-3B att-only"
evaluate(specialist, test, size=len(test))
print(f"parse failures: {failures} / {len(test)}")
leaderboard()

### Experiments

- **The sweep variants** (`5_qlora_finetune_att_only.ipynb`, `6_qlora_finetune_bs16.ipynb`):
  same code, different knobs.
- **Rank sweep**: r = 8 / 32 / 64 at matched steps.
- **Summaries vs full notes** (`-summaries` dataset from the earlier notebook).
- **Add `points` to the prompt** and watch the fine-tune coast -- the same leakage the baselines see.
- **Bigger base**: an 8B model in 4-bit still fits an A100. Does scale beat data curation here?